# 05 - Errors and important genes

A good score isn't enough. Two questions:

1. **What does the model get wrong?** Which cell types does it confuse with which?
2. **Did it learn real biology, or noise?** Which genes does each model rely on, and are they known marker genes (notebook 04)?

Steps: load data -> most common mistakes -> logistic regression's top genes -> random forest's top genes -> compare to markers.

In [ ]:
# Load the tools I need
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Same data and the same 5 folds as notebook 03, so results line up
adata = sc.datasets.pbmc68k_reduced()
X = adata.X
y = adata.obs["bulk_labels"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## The most common mistakes

We use the best model from notebook 03, logistic regression. `cross_val_predict` gives each cell one prediction, made by the fold where that cell was held out, so the model never saw it during training.

In [ ]:
# One held-out prediction per cell, same as notebook 03
model = LogisticRegression(max_iter=1000, class_weight="balanced")
predicted = cross_val_predict(model, X, y, cv=cv)

# Table of true vs predicted, then keep only the wrong ones
results = pd.DataFrame({"true": y.values, "predicted": predicted})
mistakes = results[results["true"] != results["predicted"]]

print("Total mistakes:", len(mistakes), "of", len(results))

# Count each (true, predicted) pair and show the top 5
mistakes.value_counts().head(5)

## Logistic regression: top genes per cell type

Logistic regression gives every gene a **weight** (coefficient) for every cell type. A big positive weight means "high expression of this gene pushes the vote toward this type."

Here we fit on all cells, because the goal is to explain the model, not to score it.

In [ ]:
# Fit on ALL cells
model.fit(X, y)

# coef_ has one row per cell type and one column per gene
weights = pd.DataFrame(model.coef_, index=model.classes_, columns=adata.var_names)

# For each type, the 5 genes with the largest positive weight
top_genes = {cell_type: ", ".join(weights.loc[cell_type].nlargest(5).index)
             for cell_type in model.classes_}
pd.Series(top_genes, name="top 5 genes").to_frame()

## Random forest: the 20 most useful genes

A random forest scores each gene by how much it helped the trees split cells correctly (**feature importance**). Unlike logistic regression, it gives one score per gene, not one per cell type.

In [ ]:
# Same settings as notebook 03, fit on all cells
forest = RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                random_state=42, n_jobs=-1)
forest.fit(X, y)

# One importance score per gene; keep the top 20
importance = pd.Series(forest.feature_importances_, index=adata.var_names).nlargest(20)

# Horizontal bars, most important at the top
importance.sort_values().plot.barh(figsize=(6, 7))
plt.xlabel("Importance")
plt.title("Random forest: 20 most useful genes")
plt.show()

In [ ]:
# Which of these are the textbook markers from notebook 04?
marker_list = ["CD3E", "CD8A", "IL7R", "CCR7", "MS4A1", "CD79A", "LYZ",
               "FCGR3A", "NKG7", "GNLY", "FCER1A", "CST3"]
print("Known markers in the random forest's top 20:",
      [gene for gene in importance.index if gene in marker_list])

## What I learned

_(Fill this in after reading the results above.)_

- Are the top mistakes between closely related cell types, or random?
- Do the models' top genes match the known markers from notebook 04?
- Which cell types have top genes that look like noise, and what does that say about those labels?